# 00 — Setup & Basics

**Time**: ~5 minutes

This notebook gets you up and running with Azure Document Intelligence. You'll:

1. Install the Python SDK
2. Configure authentication (API key & Entra ID)
3. Create your first client and verify the connection
4. Understand the two client classes

---

## Prerequisites

- Python 3.8+
- An Azure Document Intelligence resource ([create one here](https://portal.azure.com/#create/Microsoft.CognitiveServicesFormRecognizer))
- Your **endpoint** and **API key** from the Azure Portal

## Step 1 — Install the SDK

The Python SDK package is `azure-ai-documentintelligence`. Version 1.0.2 targets the v4.0 GA API (2024-11-30).

In [ ]:
# Install the SDK and dependencies (run once)
%pip install azure-ai-documentintelligence==1.0.2 azure-identity python-dotenv --quiet

## Step 2 — Load Your Credentials

We use a `.env` file to keep credentials out of code. Copy `.env.example` to `.env` and fill in your values.

```
DOCUMENTINTELLIGENCE_ENDPOINT=https://<your-resource>.cognitiveservices.azure.com/
DOCUMENTINTELLIGENCE_API_KEY=<your-key>
```

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

# Load credentials from .env file (looks in parent directory too)
load_dotenv(dotenv_path=os.path.join("..", ".env"))

endpoint = os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"]
_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

print(f"Endpoint: {endpoint}")
if _api_key:
    print(f"Auth: API key (...{_api_key[-4:]})")
else:
    print("Auth: DefaultAzureCredential (Entra ID)")

## Step 3 — Authentication Options

Azure Document Intelligence supports two authentication methods:

| Method | Best For | Security Level |
|---|---|---|
| **API Key** (`AzureKeyCredential`) | Development, quickstarts | Good — rotate keys regularly |
| **Entra ID** (`DefaultAzureCredential`) | Production workloads | Best — no secrets in code |

### Option A: API Key Authentication (simple, great for getting started)

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient

# Create client (uses API key if set, otherwise DefaultAzureCredential)
client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=credential
)

print("✅ DocumentIntelligenceClient created")

### Option B: Entra ID Authentication (recommended for production)

Uses `DefaultAzureCredential` which automatically tries multiple auth methods:
- Environment variables → Managed Identity → Azure CLI → Interactive browser

Requires a **custom subdomain** endpoint (not a regional endpoint) and the `Cognitive Services User` role assigned.

In [ ]:
# Uncomment to use Entra ID authentication:

# from azure.identity import DefaultAzureCredential
# from azure.ai.documentintelligence import DocumentIntelligenceClient
#
# client_entra = DocumentIntelligenceClient(
#     endpoint=endpoint,
#     credential=DefaultAzureCredential()
# )
# print("✅ Client created with Entra ID authentication")

## Step 4 — Verify Your Connection

Let's make a simple API call to confirm everything works. We'll analyze a small document using the `prebuilt-read` model.

In [ ]:
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult

# Use a simple public sample document
sample_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

# Analyze with the Read model
poller = client.begin_analyze_document(
    "prebuilt-read",
    AnalyzeDocumentRequest(url_source=sample_url)
)
result: AnalyzeResult = poller.result()

print(f"✅ Connection successful!")
print(f"   API version: {result.api_version}")
print(f"   Model ID: {result.model_id}")
print(f"   Pages analyzed: {len(result.pages)}")
print(f"   Content length: {len(result.content)} characters")

## Step 5 — Understanding the Two Client Classes

The SDK provides two client classes for different tasks:

| Client | Purpose | Key Methods |
|---|---|---|
| `DocumentIntelligenceClient` | **Analyze documents** — use prebuilt or custom models to extract data | `begin_analyze_document()`, `begin_classify_document()` |
| `DocumentIntelligenceAdministrationClient` | **Manage models** — build, list, copy, delete custom models | `begin_build_document_model()`, `list_models()`, `delete_model()` |

You already created a `DocumentIntelligenceClient` above. Here's how to create the admin client:

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceAdministrationClient

admin_client = DocumentIntelligenceAdministrationClient(
    endpoint=endpoint,
    credential=credential
)

# Check your resource details
resource_info = admin_client.get_resource_details()

print(f"✅ Admin client connected")
print(f"   Custom models: {resource_info.custom_document_models.count} / {resource_info.custom_document_models.limit}")

## Step 6 — Understanding the Analysis Flow

Every Document Intelligence analysis follows this pattern:

```
1. Submit document  →  begin_analyze_document(model_id, document)
2. Poll for result  →  poller.result()  (SDK handles polling automatically)
3. Process result   →  Access result.pages, result.documents, result.tables, etc.
```

The result object (`AnalyzeResult`) contains:

| Property | Description |
|---|---|
| `content` | Full extracted text as a string |
| `pages` | Per-page details: words, lines, selection marks |
| `tables` | Extracted tables with rows, columns, cells |
| `documents` | Extracted fields (for prebuilt/custom models) |
| `styles` | Text styles (handwritten detection, font info) |
| `paragraphs` | Semantic paragraphs with roles (title, header, etc.) |

In [ ]:
# Quick peek at the result structure from our earlier analysis
print("Result properties:")
print(f"  - content (first 200 chars): {result.content[:200]}...")
print(f"  - pages: {len(result.pages)} page(s)")
print(f"  - tables: {len(result.tables) if result.tables else 0} table(s)")
print(f"  - paragraphs: {len(result.paragraphs) if result.paragraphs else 0} paragraph(s)")
print(f"  - styles: {len(result.styles) if result.styles else 0} style(s)")

---

## ✅ You're All Set!

Your environment is configured and working. Now explore the features:

| Next Step | Notebook |
|---|---|
| Extract text from scanned claims | [01-read-model.ipynb](01-read-model.ipynb) |
| Extract tables from policy docs | [02-layout-model.ipynb](02-layout-model.ipynb) |
| Process invoices for reimbursement | [03-prebuilt-invoice.ipynb](03-prebuilt-invoice.ipynb) |
| Set up a human review workflow | [10-human-in-the-loop.ipynb](10-human-in-the-loop.ipynb) |